In [ ]:
import numpy as np
from time import time as tm

# import stuff from our own creations
from default_params import *
from utils import *
from NetworkModels import *

np.random.seed(1)

runsims=True

In [ ]:

# Duration of simulation in ms
T = 750
dt = 0.1
time = np.arange(0,T,dt)
Nt = len(time)

# Burn-in period for compring time averages and rates
Tburn = 250

dtRecord = 1
timeRecord = np.arange(0,T,dtRecord)
NtRecord = len(timeRecord)
Nburn = int(Tburn/dtRecord)

# Disctretized time
time = np.arange(0,T,dt)
Nt = len(time)
Nburn = int(Tburn/dt)


## run sims for different N and c for current-based andconductance-based synapses
N = 5000
c = 0
rxs = [10/1000, 15/1000]
numtrials = 50

# File name for saving data
filename = './Data/MultiTrialData.npy'



In [ ]:
IeRecord = []
IiRecord = []


Ne = int(N*.8)
Ni = N-Ne
Nx = int(N*qx)

if runsims:
  # Run sim
  storeSpikeCountsCond = np.zeros((numtrials, len(rxs), Ne))
  storeSpikeCounts = np.zeros((numtrials, len(rxs), Ne))

  Ee = NeuronParamsCond['Ee']
  Ei = NeuronParamsCond['Ei']
  EL = NeuronParamsCond['EL']

  gL = NeuronParamsCond['gL']
  Cm = NeuronParamsCond['Cm']

  NeuronParams = NeuronParamsCond.copy()
  del NeuronParams['Ee']
  del NeuronParams['Ei']

  JmCond = np.array([[jeeCond, jeiCond], [jieCond, jiiCond]])/np.sqrt(N)
  JxmCond = np.array([[jexCond], [jixCond]])/np.sqrt(N)

  Jm = np.array([[jee, jei], [jie, jii]])/np.sqrt(N)
  Jxm = np.array([[jex], [jix]])/np.sqrt(N)

  #########
  #### Conductance-based sim
  #########
  # Really, we should be able to do this outside of the c loop
  # but this is better because we can delete the JCond's before
  # generating the J's to reduce total mem usage
  t0=tm()
  #print('starting cond J',N)
  JCond = GetBlockErdosRenyi((Ne,Ni),JmCond,P)
  JxCond = GetBlockErdosRenyi((Nx,),JxmCond,Px,(Ne,Ni))
  tJ = tm()-t0
  #print('Time for cond J:',tJ,'s',N)

  J = GetBlockErdosRenyi((Ne,Ni),Jm,P)
  Jx = GetBlockErdosRenyi((Nx,),Jxm,Px,(Ne,Ni))

  maxns=int(N*(30/1000)*T)

  t00=tm()
  for iii in range(numtrials):
    for n in range(len(rxs)):
      rx = rxs[n]
      t0t=tm()
      # Max number of spikes

      # Set params and variables that depend on c (just Sx)
      t0=tm()
      #print('starting Sx',N)
      Sx = PoissonProcess(rx,Nt,dt,Nx,c,rep='sparse',taujitter=taujitter)
      tSx = tm()-t0
      #print('Time for cond Sx:',tSx,'s')


      t0 = tm()
      #print('starting cond sim',N)
      se,si,Recording=EIFNetworkCond(JCond, Sx, JxCond, Ne, NeuronParamsCond, tau, Nt, dt, maxns, IeRecord, IiRecord, dtRecord)
      tSim2=tm()-t0
      #print('Time for cond sim:',tSim2,'s')

      spikecounts = np.histogram(se[1,se[0,:]>Tburn],bins=np.arange(0,Ne+1,1))[0]
      storeSpikeCountsCond[iii,n] = spikecounts


      #########
      #### Naive current-based sim
      #########
      t0 = tm()
      se,si,Recording=EIFNetworkCurrent(J, Sx, Jx, Ne, NeuronParams, tau, Nt, dt, maxns, IeRecord, IiRecord, dtRecord)
      tSim2=tm()-t0
      #print('Time for curr sim:',tSim2,'s')

      spikecounts = np.histogram(se[1,se[0,:]>Tburn],bins=np.arange(0,Ne+1,1))[0]
      storeSpikeCounts[iii,n] = spikecounts

    print('Trial %d done in %.2f s'%(iii+1,tm()-t0t))

  totaltime = (tm()-t00)
  print('Total time:',totaltime/60.0,'min')


  ########################################################### save data ###########################################################

  data = {}
  data['T']=T
  data['rxs']=rxs
  data['storeSpikeCountsCond']=storeSpikeCountsCond
  data['storeSpikeCounts']=storeSpikeCounts
  data['time']=time
  data['timeRecord']=timeRecord
  data['numtrials']=numtrials
  data['NeuronParamsCond']=NeuronParamsCond
  data['JmCond']=JmCond
  data['JxmCond']=JxmCond

  data['rBal']=rBal

  np.save(filename, data)


else:
  ########################################################### load data ###########################################################
  saved_data = np.load(filename, allow_pickle=True).item()
  T = saved_data['T']
  rxs = saved_data['rxs']
  storeSpikeCountsCond = saved_data['storeSpikeCountsCond']
  storeSpikeCounts = saved_data['storeSpikeCounts']

  time = saved_data['time']
  timeRecord = saved_data['timeRecord']
  numtrials = saved_data['numtrials']
  NeuronParamsCond = saved_data['NeuronParamsCond']

  rBal = saved_data['rBal']
  #JmCond = saved_data['JmCond']
  #JxmCond = saved_data['JxmCond']

In [ ]:
print(storeSpikeCounts.shape)

trial_threshold = 10

spikecounts = storeSpikeCounts[:,0,:]
# Find neurons (columns) that have at least 10 trials with non-zero spike counts
valid_neuron_mask = (np.count_nonzero(spikecounts, axis=0) >= trial_threshold)
# Keep only those neurons in spikecounts
spikecounts = spikecounts[:, valid_neuron_mask]
print(spikecounts.shape[1],' of ',Ne,'neurons have at least 10 trials with non-zero spike counts')
FFcountsrx1 = np.var(spikecounts, axis=0)/np.mean(spikecounts, axis=0)
print('FF for rx1 current-based:',FFcountsrx1.mean(),'+/- ',FFcountsrx1.std()/np.sqrt(spikecounts.shape[1]))

rates = np.mean(spikecounts,axis=0)
print('rate:',rates.mean())

corr_mat = np.corrcoef(spikecounts.T)
upper_triangle_indices = np.triu_indices_from(corr_mat, k=1)
mean_corr_r1 = corr_mat[upper_triangle_indices].mean()
print('mean corr for rx1 current-based:',mean_corr_r1)


spikecounts = storeSpikeCounts[:,1,:]
valid_neuron_mask = (np.count_nonzero(spikecounts, axis=0) >= trial_threshold)
spikecounts = spikecounts[:, valid_neuron_mask]
print(spikecounts.shape[1],' of ',Ne,'neurons have at least 10 trials with non-zero spike counts')
FFcountsrx2 = np.var(spikecounts, axis=0)/np.mean(spikecounts, axis=0)
print('FF for rx2 current-based:',FFcountsrx2.mean(),'+/- ',FFcountsrx2.std()/np.sqrt(spikecounts.shape[1]))

rates = np.mean(spikecounts,axis=0)
print('rate:',rates.mean())

corr_mat = np.corrcoef(spikecounts.T)
upper_triangle_indices = np.triu_indices_from(corr_mat, k=1)
mean_corr_r2 = corr_mat[upper_triangle_indices].mean()
print('mean corr for rx2 current-based:',mean_corr_r2)

spikecounts = storeSpikeCountsCond[:,0,:]
valid_neuron_mask = (np.count_nonzero(spikecounts, axis=0) >= trial_threshold)
spikecounts = spikecounts[:, valid_neuron_mask]
print(spikecounts.shape[1],' of ',Ne,'neurons have at least 10 trials with non-zero spike counts')
FFcountsrx1Cond = np.var(spikecounts, axis=0)/np.mean(spikecounts, axis=0)
print('FF for rx1 conductance-based:',FFcountsrx1Cond.mean(),'+/- ',FFcountsrx1Cond.std()/np.sqrt(spikecounts.shape[1]))

rates = np.mean(spikecounts,axis=0)
print('rate:',rates.mean())

corr_mat = np.corrcoef(spikecounts.T)
upper_triangle_indices = np.triu_indices_from(corr_mat, k=1)
mean_corr_r1Cond = corr_mat[upper_triangle_indices].mean()
print('mean corr for rx1 conductance-based:',mean_corr_r1Cond)

spikecounts = storeSpikeCountsCond[:,1,:]
valid_neuron_mask = (np.count_nonzero(spikecounts, axis=0) >= trial_threshold)
spikecounts = spikecounts[:, valid_neuron_mask]
print(spikecounts.shape[1],' of ',Ne,'neurons have at least 10 trials with non-zero spike counts')
FFcountsrx2Cond = np.var(spikecounts, axis=0)/np.mean(spikecounts, axis=0)
print('FF for rx2 conductance-based:',FFcountsrx2Cond.mean(),'+/- ',FFcountsrx2Cond.std()/np.sqrt(spikecounts.shape[1]))

rates = np.mean(spikecounts,axis=0)
print('rate:',rates.mean())

corr_mat = np.corrcoef(spikecounts.T)
upper_triangle_indices = np.triu_indices_from(corr_mat, k=1)
mean_corr_r2Cond = corr_mat[upper_triangle_indices].mean()
print('mean corr for rx2 conductance-based:',mean_corr_r2Cond)



In [ ]:
iii